# RAG Evaluation Notebook (Know Your Food Guide)

### Import dependencies and set environment

This cell imports all required libraries, resolves the project path, and loads `OPENAI_API_KEY` so later cells can run without repeated setup.

In [2]:
import json
import os
import sys
from pathlib import Path
from typing import Dict, List

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'eval' else Path.cwd()
if (PROJECT_ROOT / 'rag').exists() is False:
    PROJECT_ROOT = Path('/Users/josietamburello/Desktop/NumPy/AI Engineering Lab/Know Your Food Guide')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings as LCOpenAIEmbeddings
from rag.index import all_splits
from rag.rag_chain import graph
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import llm_factory
from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness, DiscreteMetric

load_dotenv(PROJECT_ROOT / '.env')
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY missing. Add it to .env or set env var.'

print('Project root:', PROJECT_ROOT)
print('OPENAI_API_KEY loaded:', bool(os.getenv('OPENAI_API_KEY')))

100%|██████████| 8/8 [00:00<00:00, 2590.68it/s]


Split source docs into 23 chunks.
['8987f843-8e2d-42b6-9991-dbff9de59723', '328545f9-c2f3-4a48-ad22-679d06eb23ca', 'c9cdb953-0330-4d16-be43-6999fbc8a5f6']
Project root: /Users/josietamburello/Desktop/NumPy/AI Engineering Lab/Know Your Food Guide
OPENAI_API_KEY loaded: True


/var/folders/t4/tmhk5qkd4tq9kk98g0s9x0640000gn/T/ipykernel_81273/693221509.py:22: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness, DiscreteMetric
/var/folders/t4/tmhk5qkd4tq9kk98g0s9x0640000gn/T/ipykernel_81273/693221509.py:22: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness, DiscreteMetric
/var/folders/t4/tmhk5qkd4tq9kk98g0s9x0640000gn/T/ipykernel_81273/693221509.py:22: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be remov

### Initialize OpenAI client and judge LLM

This cell verifies API connectivity and creates the judge model used by RAGAs metrics.

In [3]:
MODEL = 'gpt-4o-mini'
client = OpenAI()

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': "Say 'connection OK' in exactly two words."}],
)
print('API check:', response.choices[0].message.content)

llm = llm_factory(MODEL, client=client)

API check: Connection OK.


## RAGAs Metrics

RAGAs uses LLM-as-a-judge to evaluate the full RAG pipeline with four metrics:

| Metric | What it measures | Why it matters |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? | Catches hallucination |
| **Answer Relevancy** | Does the answer address the actual question? | Catches off-topic responses |
| **Context Precision** | Was the retrieved context relevant? | Evaluates retrieval quality |
| **Context Recall** | Did retrieval capture all needed information? | Catches missing context |

### Define pipeline runner and load eval questions

This cell defines helper functions and loads `ragas_evalset.jsonl`. It also applies `MAX_SAMPLES` to keep fast-mode runs short.

In [4]:
def _norm_source(pathish: str) -> str:
    return str(pathish).replace('\\', '/').split('/')[-1]

def run_pipeline(question: str) -> dict:
    result = {}
    for update in graph.stream({'question': question}, stream_mode='updates'):
        if 'retrieve_node' in update:
            result.update(update['retrieve_node'])
        if 'generate' in update:
            result.update(update['generate'])
    docs = result.get('context') or []
    return {
        'answer': str(result.get('answer', '') or ''),
        'contexts': [str(d.page_content) for d in docs],
        'sources': [_norm_source(str(d.metadata.get('source', ''))) for d in docs],
    }

evalset_path = PROJECT_ROOT / 'eval' / 'ragas_evalset.jsonl'
rows = [json.loads(line) for line in evalset_path.read_text(encoding='utf-8').splitlines() if line.strip()]

# Fast mode: evaluate a smaller subset first.
MAX_SAMPLES = 6
rows = rows[:MAX_SAMPLES]

print('Indexed chunks:', len(all_splits))
print('Eval rows (active):', len(rows))

Indexed chunks: 23
Eval rows (active): 6


### Build evaluation dataset from pipeline outputs

This cell runs your RAG pipeline for each eval question, then stores answers and retrieved contexts in RAGAs sample objects.

In [5]:
samples = []
retrieval_rows = []
for r in rows:
    out = run_pipeline(r['question'])
    samples.append(
        SingleTurnSample(
            user_input=r['question'],
            retrieved_contexts=out['contexts'],
            response=out['answer'],
            reference=r.get('reference', ''),
        )
    )
    retrieval_rows.append({
        'question': r['question'],
        'difficulty': r.get('difficulty', ''),
        'retrieved_sources': out['sources'],
        'expected_sources': [_norm_source(x) for x in r.get('expected_sources', [])],
    })

dataset = EvaluationDataset(samples=samples)
pd.DataFrame(retrieval_rows).head()

/Users/josietamburello/Desktop/NumPy/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GuidedAnswer(used_guides=...may be more processed.'), input_type=GuidedAnswer])
  return self.__pydantic_serializer__.to_python(
/Users/josietamburello/Desktop/NumPy/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GuidedAnswer(used_guides=...y mean better quality."), input_type=GuidedAnswer])
  return self.__pydantic_serializer__.to_python(
/Users/josietamburello/Desktop/NumPy/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized v

,question,difficulty,retrieved_sources,expected_sources
0,How can I spot high-quality honey at the super...,easy,"[honey.md, honey.md, honey.md, olive_oil.md, b...",[honey.md]
1,What should I check on olive oil labels before...,easy,"[olive_oil.md, olive_oil.md, olive_oil.md, tab...",[olive_oil.md]
2,How do I verify Parmigiano Reggiano is authentic?,easy,"[parmigiano.md, parmigiano.md, parmigiano.md, ...",[parmigiano.md]
3,What does the egg shell code tell me when shop...,easy,"[eggs.md, eggs.md, eggs.md, parmigiano.md, par...",[eggs.md]
4,What should I check when buying balsamic vineg...,easy,"[balsamic_vinegar.md, balsamic_vinegar.md, bal...",[balsamic_vinegar.md]


### Compute retrieval metrics

This cell calculates recall/precision-style retrieval scores from expected vs retrieved sources. It is included to evaluate retriever quality separately from generation quality.

In [6]:
def compute_recall_at_k(retrieved_ids, ground_truth_ids, k=5):
    r = set(retrieved_ids[:k]); g = set(ground_truth_ids)
    return 0.0 if not g else len(r & g) / len(g)

def compute_precision_at_k(retrieved_ids, ground_truth_ids, k=5):
    r = set(retrieved_ids[:k]); g = set(ground_truth_ids)
    return len(r & g) / k

retrieval_eval = []
for row in retrieval_rows:
    retrieved = row['retrieved_sources']
    expected = row['expected_sources']
    if len(expected) <= 5:
        metric = 'recall@5'
        score = compute_recall_at_k(retrieved, expected, 5)
    else:
        metric = 'precision@5'
        score = compute_precision_at_k(retrieved, expected, 5)
    retrieval_eval.append({
        'question': row['question'],
        'difficulty': row['difficulty'],
        'metric': metric,
        'score': score,
        'retrieved_top_5': ', '.join(retrieved[:5]),
        'expected_sources': ', '.join(expected),
    })

retrieval_df = pd.DataFrame(retrieval_eval)
display(retrieval_df)
print(retrieval_df[['difficulty','metric','score']].groupby(['difficulty','metric']).mean())

,question,difficulty,metric,score,retrieved_top_5,expected_sources
0,How can I spot high-quality honey at the super...,easy,recall@5,1.0,"honey.md, honey.md, honey.md, olive_oil.md, ba...",honey.md
1,What should I check on olive oil labels before...,easy,recall@5,1.0,"olive_oil.md, olive_oil.md, olive_oil.md, tabl...",olive_oil.md
2,How do I verify Parmigiano Reggiano is authentic?,easy,recall@5,1.0,"parmigiano.md, parmigiano.md, parmigiano.md, b...",parmigiano.md
3,What does the egg shell code tell me when shop...,easy,recall@5,1.0,"eggs.md, eggs.md, eggs.md, parmigiano.md, parm...",eggs.md
4,What should I check when buying balsamic vineg...,easy,recall@5,1.0,"balsamic_vinegar.md, balsamic_vinegar.md, bals...",balsamic_vinegar.md
5,How do I choose better table olives from jars ...,easy,recall@5,1.0,"table_olives.md, table_olives.md, olive_oil.md...",table_olives.md


                     score
difficulty metric         
easy       recall@5    1.0


### Run RAGAs metrics (fast by default)

This cell executes the LLM-as-judge evaluation. It defaults to a single metric (`Faithfulness`) to keep runtime and API cost low; you can enable full metrics when needed.

In [ ]:
# Fast mode defaults: run only Faithfulness first.
RUN_FULL_METRICS = False

if RUN_FULL_METRICS:
    ragas_embeddings = LangchainEmbeddingsWrapper(LCOpenAIEmbeddings(model='text-embedding-3-small'))
    metrics = [
        Faithfulness(llm=llm),
        AnswerRelevancy(llm=llm, embeddings=ragas_embeddings, strictness=1),
        ContextPrecision(llm=llm),
        ContextRecall(llm=llm),
    ]
else:
    metrics = [Faithfulness(llm=llm)]

print(f'Running RAGAs evaluation with {len(metrics)} metric(s)...')
result = evaluate(dataset=dataset, metrics=metrics)
print(result)

Running RAGAs evaluation with 1 metric(s)...


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

### Convert results to a table

This cell turns the RAGAs result object into a pandas DataFrame for easy inspection and comparison across questions.

In [8]:
df = result.to_pandas()
show_cols = [c for c in ['user_input','faithfulness','answer_relevancy','context_precision','context_recall'] if c in df.columns]
print(df[show_cols].to_string(index=False))
print('\nTip: set RUN_FULL_METRICS = True to run all four metrics (slower).')

                                                user_input  faithfulness
     How can I spot high-quality honey at the supermarket?      1.000000
    What should I check on olive oil labels before buying?      1.000000
         How do I verify Parmigiano Reggiano is authentic?      0.916667
       What does the egg shell code tell me when shopping?      1.000000
What should I check when buying balsamic vinegar in store?      1.000000
    How do I choose better table olives from jars or tins?      1.000000

Tip: set RUN_FULL_METRICS = True to run all four metrics (slower).


### Save outputs

This cell writes evaluation outputs to CSV files under `eval/results`. It is placed at the end so all computed tables exist before export.

In [10]:
results_dir = PROJECT_ROOT / 'eval' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
retrieval_out = results_dir / 'retrieval_eval.csv'
ragas_out = results_dir / 'ragas_eval.csv'

ragas_frame = globals().get('ragas_df', globals().get('df'))
if ragas_frame is None:
    raise NameError("No RAGAs dataframe found. Run the cell that creates 'df' (or 'ragas_df') first.")

retrieval_df.to_csv(retrieval_out, index=False)
ragas_frame.to_csv(ragas_out, index=False)
print('Saved:', retrieval_out)
print('Saved:', ragas_out)

Saved: /Users/josietamburello/Desktop/NumPy/AI Engineering Lab/Know Your Food Guide/eval/results/retrieval_eval.csv
Saved: /Users/josietamburello/Desktop/NumPy/AI Engineering Lab/Know Your Food Guide/eval/results/ragas_eval.csv
